# Predicting Peak Electricity Demand in Massachusetts
## Part 3 of 3: Feature Engineering & Modeling

This notebook loads the cleaned dataset from Part 1, engineers predictive features,
compares four forecasting approaches of increasing sophistication (baseline → SARIMAX →
XGBoost/LightGBM → ensemble), and uses SHAP to explain what conditions drive extreme
peak demand in winter and summer.

## 3.1 Load the Cleaned Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error, mean_squared_error

df_combined = pd.read_csv("isone_full_dataset.csv", index_col="period", parse_dates=True)
target = "Demand (MW)"

print(df_combined.shape)
df_combined.head()

## 3.2 Feature Engineering

- **Lag features** (1h, 24h, 168h/1 week): capture demand's short-term persistence and
  daily/weekly cycles.
- **Rolling average** (24h): smooths short-term noise.
- **Heating/cooling degree**: quantifies deviation from a comfortable baseline temperature,
  the physical driver of heating and AC load.
- **Net load**: engineered for transparency (demand minus renewable generation) but
  *excluded* from the final feature set — see note below.

In [ ]:
df_combined["load_lag_1h"] = df_combined[target].shift(1)
df_combined["load_lag_24h"] = df_combined[target].shift(24)
df_combined["load_lag_168h"] = df_combined[target].shift(168)
df_combined["load_roll_mean_24h"] = df_combined[target].rolling(24).mean()

df_combined["cooling_degree"] = (df_combined["Temperature (°C)"] - 18).clip(lower=0)
df_combined["heating_degree"] = (18 - df_combined["Temperature (°C)"]).clip(lower=0)

# Engineered for transparency, NOT included in feature_cols below (see markdown note)
df_combined["net_load"] = df_combined[target] - (
    df_combined["Solar Generation (MWh)"] + df_combined["Wind Generation (MWh)"]
)

df_combined = df_combined.dropna()
print(df_combined.shape)

> **Note:** `net_load` is derived directly from the target variable
> (`Demand - Solar - Wind`), so including it as a feature risks inflating apparent model
> performance without adding genuine predictive insight (a form of data leakage). It is
> computed here for transparency but deliberately excluded from `feature_cols` below.

## 3.3 Train/Test Split

In [ ]:
df_combined = df_combined.sort_index()

split_date = df_combined.index.max() - pd.Timedelta(days=90)
train = df_combined[df_combined.index <= split_date]
test = df_combined[df_combined.index > split_date]

feature_cols = [
    "hour", "dayofweek", "month", "is_weekend", "is_holiday",
    "Temperature (°C)", "Humidity (%)", "Wind Speed (m/s)", "Solar Radiation (W/m²)",
    "Solar Generation (MWh)", "Wind Generation (MWh)",
    "load_lag_1h", "load_lag_24h", "load_lag_168h", "load_roll_mean_24h",
    "cooling_degree", "heating_degree"
]

print(f"Train: {train.shape[0]} rows, {train.index.min()} to {train.index.max()}")
print(f"Test:  {test.shape[0]} rows, {test.index.min()} to {test.index.max()}")

## 3.4 Baseline: Seasonal Historical Average

In [ ]:
seasonal_avg = train.groupby(["dayofweek", "hour"])[target].mean()

test = test.copy()
test["baseline_pred"] = test.apply(
    lambda row: seasonal_avg.loc[row["dayofweek"], row["hour"]], axis=1
)

mae = mean_absolute_error(test[target], test["baseline_pred"])
rmse = np.sqrt(mean_squared_error(test[target], test["baseline_pred"]))
mape = np.mean(np.abs((test[target] - test["baseline_pred"]) / test[target])) * 100

print(f"Baseline — MAE: {mae:.1f} MW, RMSE: {rmse:.1f} MW, MAPE: {mape:.2f}%")

## 3.5 Statistical Model: SARIMAX

SARIMAX is given only weather-derived exogenous variables, by design — it is not intended
to be a fully apples-to-apples comparison with the ML models (see Limitations at the end
of this notebook). Tested at two forecast horizons to illustrate how ARIMA-family models
degrade over longer, unsupervised forecast horizons.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

exog_cols = ["Temperature (°C)", "cooling_degree", "heating_degree"]

# Use a recent 6-month training window for tractable fit time
train_recent = train[train.index >= train.index.max() - pd.Timedelta(days=180)]

sarimax_model = SARIMAX(
    train_recent[target],
    exog=train_recent[exog_cols],
    order=(1, 1, 1),
    seasonal_order=(1, 0, 1, 24),
    enforce_stationarity=False,
    enforce_invertibility=False
)
sarimax_fit = sarimax_model.fit(disp=False, maxiter=100, method="lbfgs")

# Full 90-day horizon
sarimax_pred = sarimax_fit.forecast(steps=len(test), exog=test[exog_cols])
sarimax_mae = mean_absolute_error(test[target], sarimax_pred)
sarimax_rmse = np.sqrt(mean_squared_error(test[target], sarimax_pred))
sarimax_mape = np.mean(np.abs((test[target] - sarimax_pred) / test[target])) * 100
print(f"SARIMAX (90-day horizon) — MAE: {sarimax_mae:.1f} MW, RMSE: {sarimax_rmse:.1f} MW, MAPE: {sarimax_mape:.2f}%")

# Shorter 24-hour horizon, for comparison
short_forecast_steps = 24
sarimax_pred_short = sarimax_fit.forecast(steps=short_forecast_steps, exog=test[exog_cols].iloc[:short_forecast_steps])
short_mae = mean_absolute_error(test[target].iloc[:short_forecast_steps], sarimax_pred_short)
short_rmse = np.sqrt(mean_squared_error(test[target].iloc[:short_forecast_steps], sarimax_pred_short))
short_mape = np.mean(np.abs((test[target].iloc[:short_forecast_steps] - sarimax_pred_short) / test[target].iloc[:short_forecast_steps])) * 100
print(f"SARIMAX (24-hour horizon)  — MAE: {short_mae:.1f} MW, RMSE: {short_rmse:.1f} MW, MAPE: {short_mape:.2f}%")

## 3.6 Machine Learning: XGBoost

In [ ]:
import sys
!{sys.executable} -m pip install xgboost --quiet

import xgboost as xgb

xgb_model = xgb.XGBRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, random_state=42
)
xgb_model.fit(train[feature_cols], train[target])
xgb_pred = xgb_model.predict(test[feature_cols])

xgb_mae = mean_absolute_error(test[target], xgb_pred)
xgb_rmse = np.sqrt(mean_squared_error(test[target], xgb_pred))
xgb_mape = np.mean(np.abs((test[target] - xgb_pred) / test[target])) * 100
print(f"XGBoost — MAE: {xgb_mae:.1f} MW, RMSE: {xgb_rmse:.1f} MW, MAPE: {xgb_mape:.2f}%")

In [ ]:
importances = pd.Series(xgb_model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances)

## 3.7 Machine Learning: LightGBM

In [ ]:
import sys
!{sys.executable} -m pip install lightgbm --quiet

import lightgbm as lgb

lgb_model = lgb.LGBMRegressor(
    n_estimators=500, max_depth=6, learning_rate=0.05, random_state=42, verbose=-1
)
lgb_model.fit(train[feature_cols], train[target])
lgb_pred = lgb_model.predict(test[feature_cols])

lgb_mae = mean_absolute_error(test[target], lgb_pred)
lgb_rmse = np.sqrt(mean_squared_error(test[target], lgb_pred))
lgb_mape = np.mean(np.abs((test[target] - lgb_pred) / test[target])) * 100
print(f"LightGBM — MAE: {lgb_mae:.1f} MW, RMSE: {lgb_rmse:.1f} MW, MAPE: {lgb_mape:.2f}%")

## 3.8 Ensemble: Average of XGBoost + LightGBM

In [ ]:
ensemble_pred = (xgb_pred + lgb_pred) / 2

ensemble_mae = mean_absolute_error(test[target], ensemble_pred)
ensemble_rmse = np.sqrt(mean_squared_error(test[target], ensemble_pred))
ensemble_mape = np.mean(np.abs((test[target] - ensemble_pred) / test[target])) * 100
print(f"Ensemble (XGBoost + LightGBM avg) — MAE: {ensemble_mae:.1f} MW, RMSE: {ensemble_rmse:.1f} MW, MAPE: {ensemble_mape:.2f}%")

## 3.9 Model Comparison Summary

| Model | MAE (MW) | RMSE (MW) | MAPE |
|---|---|---|---|
| Baseline (seasonal avg) | 1217.0 | 1468.5 | 10.47% |
| SARIMAX (90-day horizon) | 5036.2 | 5381.1 | 43.61% |
| SARIMAX (24-hour horizon) | 1963.5 | 2189.2 | 18.29% |
| XGBoost | 151.2 | 196.0 | 1.25% |
| LightGBM | 153.5 | 199.2 | 1.27% |
| **Ensemble** | **149.1** | **193.7** | **1.23%** |

The Ensemble model performs best overall, a roughly 8.5x MAPE improvement over the
seasonal-average baseline. SARIMAX underperforms the naive baseline at both horizons —
a known limitation of ARIMA-family models over long, unsupervised forecast horizons
without access to recent ground-truth values (which the ML models get via lag features).

## 3.10 Explainability: What Drives Extreme Peak Demand? (Winter)

SHAP analysis on the top 5% highest-demand hours in the test set (October–December 2023).

In [ ]:
import sys
!{sys.executable} -m pip install shap --quiet

import shap

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(test[feature_cols])

shap.summary_plot(shap_values, test[feature_cols])

In [ ]:
threshold = test[target].quantile(0.95)
extreme_peaks = test[test[target] > threshold]
print(f"Threshold: {threshold:.0f} MW | Extreme peak hours: {len(extreme_peaks)}")

extreme_shap = explainer.shap_values(extreme_peaks[feature_cols])
shap.summary_plot(extreme_shap, extreme_peaks[feature_cols], plot_type="bar")

In [ ]:
compare_cols = ["Temperature (°C)", "hour", "is_weekend", "cooling_degree", "heating_degree"]

print("During EXTREME PEAK hours (winter):")
print(extreme_peaks[compare_cols].describe())

print("\nDuring ALL hours (winter test set):")
print(test[compare_cols].describe())

## 3.11 Explainability: What Drives Extreme Peak Demand? (Summer)

The main test window falls in fall/winter. A separate July–August window is used here to
characterize summer extreme peaks with the same already-trained XGBoost model.

In [ ]:
summer_test = df_combined[
    (df_combined.index >= "2023-07-01") & (df_combined.index <= "2023-08-31")
]
print(f"Summer test set: {summer_test.shape[0]} rows, {summer_test.index.min()} to {summer_test.index.max()}")

summer_pred = xgb_model.predict(summer_test[feature_cols])
summer_mae = mean_absolute_error(summer_test[target], summer_pred)
summer_rmse = np.sqrt(mean_squared_error(summer_test[target], summer_pred))
summer_mape = np.mean(np.abs((summer_test[target] - summer_pred) / summer_test[target])) * 100
print(f"XGBoost on Summer — MAE: {summer_mae:.1f} MW, RMSE: {summer_rmse:.1f} MW, MAPE: {summer_mape:.2f}%")

In [ ]:
summer_threshold = summer_test[target].quantile(0.95)
summer_extreme_peaks = summer_test[summer_test[target] > summer_threshold]
print(f"Summer threshold: {summer_threshold:.0f} MW | Extreme peak hours: {len(summer_extreme_peaks)}")

summer_extreme_shap = explainer.shap_values(summer_extreme_peaks[feature_cols])
shap.summary_plot(summer_extreme_shap, summer_extreme_peaks[feature_cols], plot_type="bar")

In [ ]:
print("During SUMMER EXTREME PEAK hours:")
print(summer_extreme_peaks[compare_cols].describe())

print("\nDuring ALL summer hours:")
print(summer_test[compare_cols].describe())

## 3.12 Winter vs. Summer Comparison Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
categories = ["Avg Temp (°C)", "Avg Heating\nDegree", "Avg Cooling\nDegree", "% Weekend"]

winter_peak_vals = [0.63, 17.41, 0.04, 4.6]
winter_all_vals  = [7.26, 10.92, 0.18, 28.7]

x = np.arange(len(categories))
width = 0.35
axes[0].bar(x - width/2, winter_all_vals, width, label="All Hours", color="lightblue")
axes[0].bar(x + width/2, winter_peak_vals, width, label="Extreme Peak Hours", color="darkblue")
axes[0].set_title("Winter (Oct–Dec 2023): Extreme Peaks vs. All Hours")
axes[0].set_xticks(x)
axes[0].set_xticklabels(categories, fontsize=9)
axes[0].legend()
axes[0].grid(alpha=0.3, axis="y")

summer_peak_vals = [27.94, 0.0, 9.94, 5.5]
summer_all_vals  = [22.42, 0.16, 4.58, 29.5]

axes[1].bar(x - width/2, summer_all_vals, width, label="All Hours", color="lightsalmon")
axes[1].bar(x + width/2, summer_peak_vals, width, label="Extreme Peak Hours", color="darkred")
axes[1].set_title("Summer (Jul–Aug 2023): Extreme Peaks vs. All Hours")
axes[1].set_xticks(x)
axes[1].set_xticklabels(categories, fontsize=9)
axes[1].legend()
axes[1].grid(alpha=0.3, axis="y")

plt.suptitle("What Drives Extreme Peak Demand: Winter Heating vs. Summer Cooling", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 3.13 Key Findings

In winter (Oct–Dec 2023), extreme peak hours averaged 0.6°C (vs. 7.3°C typical), with
heating-degree values roughly 60% above the seasonal average. In summer (Jul–Aug 2023),
extreme peak hours averaged 27.9°C (vs. 22.4°C typical), with cooling-degree values more
than double the seasonal average. In both seasons, extreme peaks clustered around 7 PM
and were overwhelmingly a weekday phenomenon (~95% weekday in both windows).

Across both seasons, extreme peak demand follows a consistent early-evening,
weekday-dominant pattern, driven by opposite physical mechanisms — heating load in
winter, cooling load in summer — layered on top of the daily rhythm of residential and
commercial activity.

## 3.14 Limitations

- Weather data is sourced for Boston specifically, used as a proxy for the broader MA/ISO-NE
  service area.
- SARIMAX tuning was constrained by compute time on the full multi-year dataset; a more
  exhaustively tuned model (e.g. via systematic order search) may perform better.
- SARIMAX's exogenous inputs were narrower than the ML models' by design — it received only
  weather-derived variables, while XGBoost/LightGBM had the full feature set. This is not a
  fully apples-to-apples comparison of statistical vs. ML modeling, but a comparison of each
  model family using the inputs it is best suited to handle.
- `Solar Generation (MWh)` and `Wind Generation (MWh)` contributed less than 1% combined to
  feature importance, likely because their effect on demand is already partly captured
  indirectly through weather and time-of-day features.
- The extreme-peak analysis covers two 2-month seasonal windows rather than a full year of
  rolling analysis; a complete year-round analysis would strengthen these findings further.